In [8]:
!pip install lmdb


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os, time, random, string
from types import SimpleNamespace as NS
from pathlib import Path

import torch
import torch.backends.cudnn as cudnn
import torch.nn.init as init
import torch.optim as optim
import numpy as np
from torch.utils.tensorboard import SummaryWriter

from utils import (
    CTCLabelConverter, CTCLabelConverterForBaiduWarpctc, AttnLabelConverter, Averager
)
from dataset import hierarchical_dataset, AlignCollate, Batch_Balanced_Dataset
from model import Model
import re
import torch.nn.functional as F
from nltk.metrics.distance import edit_distance


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



def validation(model, criterion, evaluation_loader, converter, opt):
    """ validation or evaluation """
    n_correct = 0
    norm_ED = 0
    length_of_data = 0
    infer_time = 0
    valid_loss_avg = Averager()

    for i, (image_tensors, labels) in enumerate(evaluation_loader):
        batch_size = image_tensors.size(0)
        length_of_data = length_of_data + batch_size
        image = image_tensors.to(device)
        # For max length prediction
        length_for_pred = torch.IntTensor([opt.batch_max_length] * batch_size).to(device)
        text_for_pred = torch.LongTensor(batch_size, opt.batch_max_length + 1).fill_(0).to(device)

        text_for_loss, length_for_loss = converter.encode(labels, batch_max_length=opt.batch_max_length)

        start_time = time.time()
        if 'CTC' in opt.Prediction:
            preds = model(image, text_for_pred)
            forward_time = time.time() - start_time

            # Calculate evaluation loss for CTC deocder.
            preds_size = torch.IntTensor([preds.size(1)] * batch_size)
            # permute 'preds' to use CTCloss format
            if opt.baiduCTC:
                cost = criterion(preds.permute(1, 0, 2), text_for_loss, preds_size, length_for_loss) / batch_size
            else:
                cost = criterion(preds.log_softmax(2).permute(1, 0, 2), text_for_loss, preds_size, length_for_loss)

            # Select max probabilty (greedy decoding) then decode index to character
            if opt.baiduCTC:
                _, preds_index = preds.max(2)
                preds_index = preds_index.view(-1)
            else:
                _, preds_index = preds.max(2)
            preds_str = converter.decode(preds_index.data, preds_size.data)
        
        else:
            preds = model(image, text_for_pred, is_train=False)
            forward_time = time.time() - start_time

            preds = preds[:, :text_for_loss.shape[1] - 1, :]
            target = text_for_loss[:, 1:]  # without [GO] Symbol
            cost = criterion(preds.contiguous().view(-1, preds.shape[-1]), target.contiguous().view(-1))

            # select max probabilty (greedy decoding) then decode index to character
            _, preds_index = preds.max(2)
            preds_str = converter.decode(preds_index, length_for_pred)
            labels = converter.decode(text_for_loss[:, 1:], length_for_loss)

        infer_time += forward_time
        valid_loss_avg.add(cost)

        # calculate accuracy & confidence score
        preds_prob = F.softmax(preds, dim=2)
        preds_max_prob, _ = preds_prob.max(dim=2)
        confidence_score_list = []
        for gt, pred, pred_max_prob in zip(labels, preds_str, preds_max_prob):
            if 'Attn' in opt.Prediction:
                gt = gt[:gt.find('[s]')]
                pred_EOS = pred.find('[s]')
                pred = pred[:pred_EOS]  # prune after "end of sentence" token ([s])
                pred_max_prob = pred_max_prob[:pred_EOS]

            # To evaluate 'case sensitive model' with alphanumeric and case insensitve setting.
            if opt.sensitive and opt.data_filtering_off:
                pred = pred.lower()
                gt = gt.lower()
                alphanumeric_case_insensitve = '0123456789abcdefghijklmnopqrstuvwxyz'
                out_of_alphanumeric_case_insensitve = f'[^{alphanumeric_case_insensitve}]'
                pred = re.sub(out_of_alphanumeric_case_insensitve, '', pred)
                gt = re.sub(out_of_alphanumeric_case_insensitve, '', gt)

            if pred == gt:
                n_correct += 1

            '''
            (old version) ICDAR2017 DOST Normalized Edit Distance https://rrc.cvc.uab.es/?ch=7&com=tasks
            "For each word we calculate the normalized edit distance to the length of the ground truth transcription."
            if len(gt) == 0:
                norm_ED += 1
            else:
                norm_ED += edit_distance(pred, gt) / len(gt)
            '''

            # ICDAR2019 Normalized Edit Distance
            if len(gt) == 0 or len(pred) == 0:
                norm_ED += 0
            elif len(gt) > len(pred):
                norm_ED += 1 - edit_distance(pred, gt) / len(gt)
            else:
                norm_ED += 1 - edit_distance(pred, gt) / len(pred)

            # calculate confidence score (= multiply of pred_max_prob)
            try:
                confidence_score = pred_max_prob.cumprod(dim=0)[-1]
            except:
                confidence_score = 0  # for empty pred case, when prune after "end of sentence" token ([s])
            confidence_score_list.append(confidence_score)
            # print(pred, gt, pred==gt, confidence_score)

    accuracy = n_correct / float(length_of_data) * 100
    norm_ED = norm_ED / float(length_of_data)  # ICDAR2019 Normalized Edit Distance

    return valid_loss_avg.val(), accuracy, norm_ED, preds_str, confidence_score_list, labels, infer_time, length_of_data

def make_opt(**overrides):
    opt = NS(
        exp_name=None,
        saved_model='',
        FT=False,
        train_data='./data/train',
        valid_data='./data/val',
        manualSeed=1111,
        workers=4,
        batch_size=192,
        num_iter=300000,
        valInterval=2000,
        adam=False,
        lr=1.0,
        beta1=0.9,
        rho=0.95,
        eps=1e-8,
        grad_clip=5.0,
        baiduCTC=False,
        select_data='MJ-ST',
        batch_ratio='0.5-0.5',
        total_data_usage_ratio='1.0',
        batch_max_length=25,
        imgH=32,
        imgW=100,
        rgb=False,
        character='0123456789abcdefghijklmnopqrstuvwxyz',
        sensitive=False,
        PAD=False,
        data_filtering_off=False,
        Transformation='TPS',
        FeatureExtraction='ResNet',
        SequenceModeling='BiLSTM',
        Prediction='Attn',
        num_fiducial=20,
        input_channel=1,
        output_channel=512,
        hidden_size=256,
    )
    for k, v in overrides.items():
        setattr(opt, k, v)
    if opt.sensitive:
        opt.character = string.printable[:-6]
    if opt.exp_name is None:
        opt.exp_name = f'{opt.Transformation}-{opt.FeatureExtraction}-{opt.SequenceModeling}-{opt.Prediction}-Seed{opt.manualSeed}'
    Path(f'./saved_models/{opt.exp_name}').mkdir(parents=True, exist_ok=True)
    Path(f'./saved_models/{opt.exp_name}/tb').mkdir(parents=True, exist_ok=True)
    return opt


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    cudnn.benchmark = True
    cudnn.deterministic = True


def init_weights_(model: torch.nn.Module):
    for name, param in model.named_parameters():
        if 'localization_fc2' in name:
            print(f'Skip {name} as it is already initialized')
            continue
        try:
            if 'bias' in name:
                init.constant_(param, 0.0)
            elif 'weight' in name:
                init.kaiming_normal_(param)
        except Exception:
            if 'weight' in name:
                param.data.fill_(1)


def build_converter_and_num_class(opt):
    if 'CTC' in opt.Prediction:
        if opt.baiduCTC:
            converter = CTCLabelConverterForBaiduWarpctc(opt.character)
        else:
            converter = CTCLabelConverter(opt.character)
    else:
        converter = AttnLabelConverter(opt.character)
    opt.num_class = len(converter.character)
    return converter


def train(opt):
    seed_everything(opt.manualSeed)
    opt.select_data = opt.select_data.split('-')
    opt.batch_ratio = opt.batch_ratio.split('-')
    if not opt.data_filtering_off:
        print('Filtering the images containing characters not in opt.character')
        print('Filtering the images whose label length > opt.batch_max_length')
    train_dataset = Batch_Balanced_Dataset(opt)
    AlignCollate_valid = AlignCollate(imgH=opt.imgH, imgW=opt.imgW, keep_ratio_with_pad=opt.PAD)
    valid_dataset, valid_dataset_log = hierarchical_dataset(root=opt.valid_data, opt=opt)
    valid_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=opt.batch_size, shuffle=True,
        num_workers=int(opt.workers), collate_fn=AlignCollate_valid, pin_memory=True
    )
    with open(f'./saved_models/{opt.exp_name}/log_dataset.txt', 'a', encoding='utf-8') as logf:
        logf.write(valid_dataset_log)
        logf.write('\n' + '-' * 80 + '\n')
    converter = build_converter_and_num_class(opt)
    if opt.rgb:
        opt.input_channel = 3
    model = Model(opt)
    print('model input parameters',
          opt.imgH, opt.imgW, opt.num_fiducial, opt.input_channel, opt.output_channel,
          opt.hidden_size, opt.num_class, opt.batch_max_length, opt.Transformation,
          opt.FeatureExtraction, opt.SequenceModeling, opt.Prediction)
    init_weights_(model)
    opt.num_gpu = torch.cuda.device_count()
    if opt.num_gpu > 1:
        print('------ Use multi-GPU setting ------')
        print('If stuck too long with multi-GPU, try --workers 0 equivalent')
        opt.workers = opt.workers * opt.num_gpu
        opt.batch_size = opt.batch_size * opt.num_gpu
    model = torch.nn.DataParallel(model).to(device)
    model.train()
    if opt.saved_model:
        print(f'loading pretrained model from {opt.saved_model}')
        sd = torch.load(opt.saved_model, map_location='cpu')
        model.load_state_dict(sd, strict=not opt.FT)
    print("Model:\n", model)
    if 'CTC' in opt.Prediction:
        if opt.baiduCTC:
            from warpctc_pytorch import CTCLoss
            criterion = CTCLoss()
        else:
            criterion = torch.nn.CTCLoss(zero_infinity=True).to(device)
    else:
        criterion = torch.nn.CrossEntropyLoss(ignore_index=0).to(device)
    loss_avg = Averager()
    filtered_parameters = [p for p in model.parameters() if p.requires_grad]
    params_num = [np.prod(p.size()) for p in filtered_parameters]
    print('Trainable params num : ', int(sum(params_num)))
    if opt.adam:
        optimizer = optim.Adam(filtered_parameters, lr=opt.lr, betas=(opt.beta1, 0.999))
    else:
        optimizer = optim.Adadelta(filtered_parameters, lr=opt.lr, rho=opt.rho, eps=opt.eps)
    print("Optimizer:\n", optimizer)
    with open(f'./saved_models/{opt.exp_name}/opt.txt', 'a', encoding='utf-8') as opt_file:
        opt_log = '------------ Options -------------\n'
        for k, v in vars(opt).items():
            opt_log += f'{k}: {v}\n'
        opt_log += '----------------------------------\n'
        print(opt_log)
        opt_file.write(opt_log)
    tb_dir = f'./saved_models/{opt.exp_name}/tb'
    writer = SummaryWriter(log_dir=tb_dir)
    writer.add_text('config/exp_name', opt.exp_name)
    start_iter = 0
    if opt.saved_model:
        try:
            start_iter = int(Path(opt.saved_model).stem.split('_')[-1])
            print(f'continue to train, start_iter: {start_iter}')
        except Exception:
            pass
    start_time = time.time()
    best_accuracy = -1.0
    best_norm_ED = -1.0
    iteration = start_iter
    log_train_path = f'./saved_models/{opt.exp_name}/log_train.txt'
    while True:
        image_tensors, labels = train_dataset.get_batch()
        image = image_tensors.to(device)
        text, length = converter.encode(labels, batch_max_length=opt.batch_max_length)
        batch_size = image.size(0)
        if 'CTC' in opt.Prediction:
            preds = model(image, text)
            preds_size = torch.IntTensor([preds.size(1)] * batch_size)
            if opt.baiduCTC:
                preds = preds.permute(1, 0, 2)
                cost = criterion(preds, text, preds_size, length) / batch_size
            else:
                preds = preds.log_softmax(2).permute(1, 0, 2)
                cost = criterion(preds, text, preds_size, length)
        else:
            preds = model(image, text[:, :-1])
            target = text[:, 1:]
            cost = criterion(preds.view(-1, preds.shape[-1]), target.contiguous().view(-1))
        model.zero_grad()
        cost.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), opt.grad_clip)
        optimizer.step()
        loss_avg.add(cost)
        step = iteration + 1
        writer.add_scalar('train/loss', float(cost.item()), step)
        writer.add_scalar('train/lr', float(optimizer.param_groups[0]['lr']), step)
        writer.add_scalar('train/batch_size', int(batch_size), step)
        if (step % opt.valInterval == 0) or (iteration == 0):
            elapsed_time = time.time() - start_time
            with open(log_train_path, 'a', encoding='utf-8') as logf:
                model.eval()
                with torch.no_grad():
                    valid_loss, current_accuracy, current_norm_ED, preds_txt, confidence_score, labels_txt, infer_time, length_of_data = validation(
                        model, criterion, valid_loader, converter, opt
                    )
                model.train()
                loss_log = f'[{step}/{opt.num_iter}] Train loss: {loss_avg.val():0.5f}, Valid loss: {valid_loss:0.5f}, Elapsed_time: {elapsed_time:0.5f}'
                loss_avg.reset()
                current_model_log = f'{"Current_accuracy":17s}: {current_accuracy:0.3f}, {"Current_norm_ED":17s}: {current_norm_ED:0.2f}'
                if current_accuracy > best_accuracy:
                    best_accuracy = current_accuracy
                    torch.save(model.state_dict(), f'./saved_models/{opt.exp_name}/best_accuracy.pth')
                if current_norm_ED > best_norm_ED:
                    best_norm_ED = current_norm_ED
                    torch.save(model.state_dict(), f'./saved_models/{opt.exp_name}/best_norm_ED.pth')
                best_model_log = f'{"Best_accuracy":17s}: {best_accuracy:0.3f}, {"Best_norm_ED":17s}: {best_norm_ED:0.2f}'
                loss_model_log = f'{loss_log}\n{current_model_log}\n{best_model_log}'
                print(loss_model_log)
                logf.write(loss_model_log + '\n')
                dashed_line = '-' * 80
                head = f'{"Ground Truth":25s} | {"Prediction":25s} | Confidence Score & T/F'
                predicted_result_log = f'{dashed_line}\n{head}\n{dashed_line}\n'
                for gt, pred, conf in zip(labels_txt[:5], preds_txt[:5], confidence_score[:5]):
                    if 'Attn' in opt.Prediction:
                        gt = gt[:gt.find('[s]')]
                        pred = pred[:pred.find('[s]')]
                    predicted_result_log += f'{gt:25s} | {pred:25s} | {conf:0.4f}\t{str(pred == gt)}\n'
                predicted_result_log += dashed_line
                print(predicted_result_log)
                logf.write(predicted_result_log + '\n')
            writer.add_scalar('valid/loss', float(valid_loss), step)
            writer.add_scalar('valid/accuracy', float(current_accuracy), step)
            writer.add_scalar('valid/norm_ED', float(current_norm_ED), step)
            writer.add_scalar('time/elapsed', float(elapsed_time), step)
            if length_of_data:
                writer.add_scalar('valid/avg_infer_time_per_sample', float(infer_time / length_of_data), step)
            writer.add_text('valid/samples', predicted_result_log, step)
            writer.flush()
        if (step % int(1e5) == 0):
            torch.save(model.state_dict(), f'./saved_models/{opt.exp_name}/iter_{step}.pth')
        if step == opt.num_iter:
            print('Training finished.')
            try:
                writer.add_hparams(
                    hparam_dict={
                        'arch/T': opt.Transformation, 'arch/FE': opt.FeatureExtraction,
                        'arch/SM': opt.SequenceModeling, 'arch/Pred': opt.Prediction,
                        'train/batch_size': opt.batch_size, 'train/lr': opt.lr,
                        'train/optimizer': 'Adam' if opt.adam else 'Adadelta',
                        'data/select_data': '-'.join(opt.select_data),
                        'data/batch_ratio': '-'.join(opt.batch_ratio),
                        'seed': opt.manualSeed
                    },
                    metric_dict={
                        'best/accuracy': float(best_accuracy),
                        'best/norm_ED': float(best_norm_ED)
                    }
                )
            except Exception:
                pass
            writer.close()
            return best_accuracy, best_norm_ED
        iteration += 1


In [10]:
import os, platform, subprocess
from pathlib import Path

def prepare_dataset_root(root, items):
    Path(root).mkdir(parents=True, exist_ok=True)
    is_win = platform.system().lower().startswith('win')
    for alias, target in items:
        link_path = Path(root) / alias
        if link_path.exists():
            continue
        if is_win:
            subprocess.run(['cmd', '/c', 'mklink', '/J', str(link_path), str(Path(target))], check=True)
        else:
            os.symlink(str(Path(target)), str(link_path))

# --- Эксперимент 1 ---
exp1_train_root = r"C:\shared\for_report4\_mix_exp1_train"
exp1_val_root   = r"C:\shared\for_report4\_mix_exp1_val"

prepare_dataset_root(exp1_train_root, [
    ("data_train_imdb", r"C:\shared\for_report4\handwrited\subset_dataset\train_lmdb"),
])
prepare_dataset_root(exp1_val_root, [
    ("data_test_imdb", r"C:\shared\for_report4\handwrited\subset_dataset\test_lmdb"),
])

opt1 = make_opt(
    exp_name='exp_orig',
    train_data=exp1_train_root,
    valid_data=exp1_val_root,
    select_data='data_train_imdb',
    batch_ratio='1.0',
    manualSeed=1111,
    workers=0,
    batch_size=128,
    num_iter=30000,
    valInterval=2000,
    adam=False,
    lr=1.0,
    Transformation='None',
    FeatureExtraction='VGG',
    SequenceModeling='BiLSTM',
    Prediction='Attn',
    imgH=32, imgW=100, rgb=False,
    character=r" !%'()*+,-./0123456789:;<=>[]^_v{\|}~§«»АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяёіѣѳѵ№",
    PAD=True,
)
best_acc_1, best_ed_1 = train(opt1)
print('EXP1 -> best_acc:', best_acc_1, 'best_norm_ED:', best_ed_1)


Filtering the images containing characters not in opt.character
Filtering the images whose label length > opt.batch_max_length
--------------------------------------------------------------------------------
dataset_root: C:\shared\for_report4\_mix_exp1_train
opt.select_data: ['data_train_imdb']
opt.batch_ratio: ['1.0']
--------------------------------------------------------------------------------
dataset_root:    C:\shared\for_report4\_mix_exp1_train	 dataset: data_train_imdb
sub-directory:	/data_train_imdb	 num samples: 40902
num total samples of data_train_imdb: 40902 x 1.0 (total_data_usage_ratio) = 40902
num samples of data_train_imdb per batch: 128 x 1.0 (batch_ratio) = 128
--------------------------------------------------------------------------------
Total_batch_size: 128 = 128
--------------------------------------------------------------------------------
dataset_root:    C:\shared\for_report4\_mix_exp1_val	 dataset: /
sub-directory:	/data_test_imdb	 num samples: 21072
No 

In [11]:
import os, platform, subprocess
from pathlib import Path

def prepare_dataset_root(root, items):
    Path(root).mkdir(parents=True, exist_ok=True)
    is_win = platform.system().lower().startswith('win')
    for alias, target in items:
        link_path = Path(root) / alias
        if link_path.exists():
            continue
        if is_win:
            subprocess.run(['cmd', '/c', 'mklink', '/J', str(link_path), str(Path(target))], check=True)
        else:
            os.symlink(str(Path(target)), str(link_path))


# --- Эксперимент 2 ---
exp2_train_root = r"C:\shared\for_report4\_mix_exp2_train"
exp2_val_root   = r"C:\shared\for_report4\_mix_exp2_val"

prepare_dataset_root(exp2_train_root, [
    ("data_train_imdb", r"C:\shared\for_report4\handwrited\subset_dataset\train_lmdb"),
    ("data_synth_train_imdb", r"C:\shared\for_report4\handwrited\subset_dataset\data_synth_train_imdb"),
])
prepare_dataset_root(exp2_val_root, [
    ("data_test_imdb", r"C:\shared\for_report4\handwrited\subset_dataset\test_lmdb"),
])


opt2 = make_opt(
    exp_name='exp_with_synth',
    train_data=exp2_train_root,
    valid_data=exp2_val_root,
    select_data='data_train_imdb-data_synth_train_imdb',
    batch_ratio='0.5-0.5',
    manualSeed=1111,
    workers=0,
    batch_size=128,
    num_iter=30000,
    valInterval=2000,
    adam=False,
    lr=1.0,
    Transformation='None',
    FeatureExtraction='VGG',
    SequenceModeling='BiLSTM',
    Prediction='Attn',
    imgH=32, imgW=100, rgb=False,
    character=r" !%'()*+,-./0123456789:;<=>[]^_v{\|}~§«»АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяёіѣѳѵ№",
    PAD=True,
)
best_acc_2, best_ed_2 = train(opt2)
print('EXP2 -> best_acc:', best_acc_2, 'best_norm_ED:', best_ed_2)

Filtering the images containing characters not in opt.character
Filtering the images whose label length > opt.batch_max_length
--------------------------------------------------------------------------------
dataset_root: C:\shared\for_report4\_mix_exp2_train
opt.select_data: ['data_train_imdb', 'data_synth_train_imdb']
opt.batch_ratio: ['0.5', '0.5']
--------------------------------------------------------------------------------
dataset_root:    C:\shared\for_report4\_mix_exp2_train	 dataset: data_train_imdb
sub-directory:	/data_train_imdb	 num samples: 40902
num total samples of data_train_imdb: 40902 x 1.0 (total_data_usage_ratio) = 40902
num samples of data_train_imdb per batch: 128 x 0.5 (batch_ratio) = 64
--------------------------------------------------------------------------------
dataset_root:    C:\shared\for_report4\_mix_exp2_train	 dataset: data_synth_train_imdb
sub-directory:	/data_synth_train_imdb	 num samples: 938202
num total samples of data_synth_train_imdb: 938202 